# Lesson 5: How LLMs Work

**Time:** ~45 minutes | **Cost:** $0 (no API calls)

## What is an LLM?

An LLM is like a **student who read the entire internet** — billions of web pages, books, articles, and code. From all that reading, it learned *patterns* in language: how sentences are structured, how ideas connect, what typically follows what.

When you ask an LLM a question, it doesn't look up the answer in a database. It **predicts the most likely text** that should come next, based on all those patterns.

```
[Your input] → [LLM] → [Predicted text]
```

**LLM** stands for **Large Language Model**. The models you've heard of — **Claude**, **GPT**, **Grok** — are all LLMs. They differ in training data, architecture, and capabilities, but they all work on the same principle: **predicting the next piece of text**.

## Tokens — How LLMs Read Text

LLMs don't read words the way you do. They break text into **tokens** — small chunks that are roughly **3/4 of a word** on average.

For example:
- `"Hello"` → 1 token
- `"unbelievable"` → might be split into `"un"` + `"believ"` + `"able"` → 3 tokens
- `"SEO"` → 1 token (common abbreviation)

**Why this matters:** Every API call costs money **per token**. When you send text to Claude and get a response, you're paying for the tokens going in (input) and the tokens coming out (output).

Rough guide:

| Text | Approx. tokens |
|------|---------------|
| A tweet (280 chars) | ~60–80 |
| A blog post (1,000 words) | ~1,300 |
| A full SEO article (2,500 words) | ~3,300 |

**Rule of thumb:** 1 word ≈ 1.33 tokens. Or flip it: 1 token ≈ 3/4 of a word.

In [ ]:
# Let's estimate token counts!
sample_texts = [
    "SEO is important for website rankings",
    "On-page SEO optimization involves improving title tags, meta descriptions, header tags, and content quality to help search engines understand your pages better.",
    "A comprehensive guide to building backlinks for your website that covers guest posting, broken link building, and digital PR strategies for improving domain authority.",
]

for text in sample_texts:
    word_count = len(text.split())
    estimated_tokens = int(word_count * 4 / 3)  # rough estimate: 1 word ≈ 1.33 tokens
    print(f'Text: "{text[:50]}..."')
    print(f"  Words: {word_count}, Estimated tokens: {estimated_tokens}")
    print()

## Context Windows — The LLM's Working Memory

An LLM's **context window** is the **maximum number of tokens** (input + output combined) it can handle in a single conversation.

| Model | Context window |
|-------|---------------|
| Claude Sonnet | 200,000 tokens |
| GPT-4o | 128,000 tokens |
| Grok | 131,072 tokens |

200,000 tokens is roughly **150,000 words** — a full-length novel.

**Why this matters for agents:** When our agent writes an SEO article, everything has to fit in that window:
- The **instructions** you gave the agent
- The **research notes** from web search
- The **article** it's writing

If the total exceeds the context window, the model cannot process it.

In [ ]:
# How much fits in Claude's context window?
context_window = 200_000  # tokens

article_tokens = 3_300        # one SEO article
research_notes_tokens = 2_000  # research for one article
instructions_tokens = 500      # agent instructions

single_task = research_notes_tokens + instructions_tokens + article_tokens
print(f"One article task uses ~{single_task:,} tokens")
print(f"That's {single_task / context_window * 100:.1f}% of the context window")
print(f"Claude could fit ~{context_window // single_task} article tasks in its window")
print()
print(f"A 50-page report (~65,000 tokens) uses {65_000 / context_window * 100:.0f}% of the window")
print(f"That still leaves room for instructions and output!")

## Training Data and Knowledge Cutoff

An LLM learns from a massive dataset of text — web pages, books, code, articles — collected up to a specific date. That date is the **knowledge cutoff**.

After that date, the model is **blind**. It has no idea what happened.

For example:
- If Claude's training data goes up to early 2025, and you ask *"What were the top SEO trends in late 2025?"* — it can't know. It might guess, or worse, make something up.
- Ask it about Google's latest algorithm update from last week? Same problem.

This is why we give agents **tools** — like web search via DataForSEO. The agent can search the live internet to get **fresh, current information** instead of relying on its outdated training data.

> **Preview:** In Lesson 09, you'll give an agent web search tools to solve this problem.

## Next Token Prediction — How LLMs Generate Text

When an LLM generates text, it follows this loop:

1. **Receive** the input text (your prompt)
2. **Predict** the single most likely next token
3. **Append** that token to the text
4. **Repeat** steps 2–3 until the response is complete

It's autocomplete on your phone, but far more sophisticated. Your phone predicts the next word; an LLM predicts the next token using billions of learned patterns.

### Temperature — The Creativity Dial

**Temperature** controls how the model picks the next token:

- **Temperature 0** — Always picks the most likely token. Output is **predictable, consistent, and factual**. Great for data extraction, structured output.
- **Temperature 1** — Gives other tokens a fair chance. Output is **more varied, creative, and sometimes surprising**. Better for creative writing, brainstorming.

```
Focused/Safe ←——— Temperature ———→ Creative/Risky
     0.0                                    1.0
```

**Connection to agents:** When you write `instructions` for an agent, you're biasing *which tokens* the model is likely to predict. Saying "be concise" makes short, direct tokens more probable. Saying "be creative and use metaphors" shifts the probabilities toward more colorful language.

In [ ]:
import random

# Simplified demo of how temperature affects word choice
# (Real LLMs use much more sophisticated math, but the idea is the same)

next_word_options = {
    "SEO is": {
        "important": 0.40,
        "essential": 0.25,
        "critical": 0.15,
        "fundamental": 0.10,
        "overrated": 0.05,
        "magical": 0.03,
        "bananas": 0.02,
    }
}

prompt = "SEO is"
words = list(next_word_options[prompt].keys())
weights = list(next_word_options[prompt].values())

print('Prompt: "SEO is ___"\n')
print("Low temperature (more focused):")
for i in range(5):
    # Low temp: amplify high-probability words
    adjusted = [w ** 2 for w in weights]
    total = sum(adjusted)
    adjusted = [w / total for w in adjusted]
    choice = random.choices(words, weights=adjusted, k=1)[0]
    print(f"  -> SEO is {choice}")

print("\nHigh temperature (more creative):")
for i in range(5):
    # High temp: flatten probabilities (more equal chances)
    adjusted = [w ** 0.5 for w in weights]
    total = sum(adjusted)
    adjusted = [w / total for w in adjusted]
    choice = random.choices(words, weights=adjusted, k=1)[0]
    print(f"  -> SEO is {choice}")

Run the cell above a few times — you'll notice:
- **Low temperature** mostly picks "important" and "essential" (the top candidates)
- **High temperature** occasionally throws in "magical" or even "bananas"

This is why temperature matters for SEO content: you want factual, consistent articles (low temperature), not creative surprises in your statistics.

## What LLMs Cannot Do

LLMs are impressive, but they have real limitations that directly affect your work.

### Hallucinations
LLMs can sound **completely confident** while being **completely wrong**. They might:
- Invent statistics that don't exist
- Cite research papers that were never written
- State outdated information as current fact

This happens because the model is *predicting plausible text*, not *looking up facts*.

### No Real-Time Knowledge
As covered above — after the training cutoff date, the model is blind. It cannot access the internet on its own (unless you give it tools).

### No Memory Between Conversations
Each time you start a new conversation, the LLM starts from scratch. It doesn't remember what you discussed yesterday. (This is why our chat system in Lesson 18 explicitly builds in memory.)

### Unreliable Math
LLMs are **not calculators**. They can make basic arithmetic errors, especially with larger numbers. Never trust an LLM to do your accounting.

---

**Why SEO teams need to care:** A generated article might include fake statistics, invented citations, or outdated claims — and it will all *sound* perfectly authoritative. This is why our pipeline puts articles in **"review" status**, not "published" — **human review is essential**, not optional.

## Lesson 5 Summary

| Term | Meaning | Why SEO teams care |
|------|---------|-------------------|
| **Token** | ~3/4 of a word, the unit LLMs process | API costs are per token |
| **Context window** | Maximum input+output size | Limits how much research fits per article |
| **Knowledge cutoff** | Date training data ends | Why agents need web search tools |
| **Temperature** | Creativity dial (0=focused, 1=creative) | Controls writing style consistency |
| **Hallucination** | Confident but wrong output | Articles need human fact-checking |
| **Next token prediction** | How LLMs generate text | Understanding this helps write better prompts |

**What you learned:**
- LLMs are pattern-matching machines that **predict the next token** — they don't "know" things
- **Tokens** are the currency of LLMs — every API call costs money per token
- **Context windows** set the limit on how much an agent can work with at once
- **Temperature** controls the balance between predictable and creative output
- LLMs **hallucinate**, have **no real-time knowledge**, and **no memory** — human review is non-negotiable

**Next lesson:** How to write effective prompts — the skill that separates a useless agent from a powerful one.

## Exercises

No code needed for the first two — think through them:

**1.** You want to generate an SEO article about *"best running shoes 2026."* Which LLM limitation makes this topic risky without tools?

<details>
<summary>Click to reveal answer</summary>

**Knowledge cutoff** — the LLM's training data likely doesn't include 2026 products. Without web search tools, it might hallucinate shoe models that don't exist.

</details>

**2.** Your generated article about *"10 SEO Statistics"* contains the claim: *"93% of online experiences begin with a search engine."* Should you trust this? Why or why not?

<details>
<summary>Click to reveal answer</summary>

**No** — hallucination risk. The LLM might have invented or misremembered this statistic. Even if it's a real stat, it could be outdated. Always verify statistics from the original source before publishing.

</details>

**3.** Estimate the token cost of generating a 2,000-word article if the model charges **$3 per million input tokens** and **$15 per million output tokens**.

Hint: 2,000 words ≈ 2,700 tokens (output). Assume ~3,000 tokens input for instructions + outline.

Write your calculation in the cell below.

In [ ]:
# Exercise: Calculate the cost estimate from question 3 above
# Input: ~3,000 tokens at $3 per million
# Output: ~2,700 tokens at $15 per million

# Your code here:
